<a href="https://colab.research.google.com/github/justorfc/Estadistica_Aplicada_con_Python_y_R_2026_2/blob/main/15_Semana_15_Modelaci%C3%B3n_Avanzada_de_Series_de_Tiempo_y_Pron%C3%B3stico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este es el Notebook de la propuesta estructurada para la **Semana 15**. Nos encontramos en la recta final del **Eje V**. En esta semana damos el paso definitivo: utilizar la memoria matemática de la serie para generar pronósticos reales (*forecasting*) y, lo más importante en ingeniería, calcular la banda de incertidumbre de nuestras predicciones antes del proyecto final.

# Semana 15: Modelación Avanzada de Series de Tiempo y Pronóstico

**Resultado de aprendizaje:** Genera pronósticos temporales utilizando modelos de suavizado exponencial y ARIMA, evaluando su desempeño predictivo e interpretando los intervalos de confianza para la toma de decisiones con incertidumbre.

---

#### Sesión 1: Suavizado Exponencial, ARIMA y la Incertidumbre (80 - 90 minutos)

**Objetivo:** Superar el modelo "Ingenuo" de la semana anterior ajustando un modelo matemático capaz de proyectar simultáneamente la tendencia y la estacionalidad, visualizando los intervalos de confianza en Python.

* **20 min - Diálogo socrático y conceptualización (Lápiz y papel):**
* *Situación:* Un gerente de distrito de riego te pide que pronostiques exactamente cuánta agua consumirá el cultivo en diciembre del próximo año. ¿Le das un solo número (ej. $120\text{ mm}$) o un rango ($105 - 135\text{ mm}$)?
* *Actividad:* Discusión sobre el "Cono de Incertidumbre". A medida que intentamos predecir más lejos en el futuro, el error posible crece. Introducción conceptual intuitiva a Holt-Winters (Suavizado Exponencial) y ARIMA (Autoregresivo Integrado de Medias Móviles).


* **45 min - Exploración en Google Colab (Python):**
* Carga del cuaderno de la semana 15.
* Ajuste de un modelo SARIMAX (ARIMA Estacional) usando `statsmodels`.
* Generación de pronósticos sobre el conjunto de validación y extracción de los Intervalos de Confianza (95%).


* **15 min - Reflexión manuscrita:**
* Interpretación técnica: ¿Por qué el intervalo de confianza se va abriendo (haciéndose más ancho) conforme avanzamos hacia los últimos meses del pronóstico?



---

#### Sesión 2: El Poder de `auto.arima` y Preparación para el Proyecto (80 - 90 minutos)

**Objetivo:** Descubrir cómo el ecosistema de R automatiza la búsqueda del mejor modelo temporal y organizar las bases para el Proyecto Integrador de la próxima semana.

* **25 min - La selección automática de modelos temporales:**
* Explicación de cómo las computadoras prueban cientos de combinaciones de parámetros (p, d, q) y utilizan el criterio AIC (visto en la Semana 8) para quedarse con el modelo que mejor pronostica sin sobreajustarse.


* **20 min - Prompts para forecasting avanzado en R:**
* Demostración de cómo instruir a la IA para utilizar la genial función `auto.arima()` del paquete `forecast` en R, la cual hace el trabajo pesado de selección matemática, y la función `autoplot()` para visualizar el pronóstico con sus bandas de sombra de manera automática.


* **40 min - Reto en Posit Cloud:**
* Los estudiantes ejecutan el flujo en su documento RMarkdown/Quarto. Comparan el MAPE de su `auto.arima` contra el MAPE del modelo Ingenuo de la Semana 14, y registran sus conclusiones finales de modelación en la bitácora de IA.



---

A continuación, el contenido listo para integrarse en las celdas de tu cuaderno de Google Colab.

---

### Celda de Texto 1

```markdown
# Semana 15: Pronóstico (Forecasting) e Intervalos de Confianza
**Asignatura:** Estadística Aplicada con Python y R  
**Programa:** Ingeniería Agrícola - Universidad de Sucre  
**Profesor:** Justo Rafael Fuentes Cuello  

---

### Situación de Interés: Prediciendo el Futuro con Incertidumbre
En la semana 14 creamos un modelo de pronóstico "Ingenuo" que repetía el pasado reciente. Hoy construiremos un modelo matemático real.

Utilizaremos la familia de modelos **ARIMA Estacional (SARIMA)**, que combina la regresión sobre los propios datos del pasado (memoria) con los ciclos estacionales. Más importante aún, como en ingeniería no existen las certezas absolutas, obligaremos al modelo a darnos un **Intervalo de Confianza del 95%**. Es decir, no solo nos dará un valor estimado, sino un rango donde hay un 95% de probabilidad matemática de que caiga el valor real.

```

### Celda de Código 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
np.random.seed(42)

# Simulamos 10 años de Demanda Hídrica (mm/mes) de una cuenca
fechas = pd.date_range(start='2014-01-01', periods=120, freq='ME')
meses_idx = np.arange(120)

# Tendencia al alza + Estacionalidad anual + Ruido aleatorio
tendencia = 0.5 * meses_idx + 100
estacionalidad = 40 * np.sin(2 * np.pi * meses_idx / 12)
ruido = np.random.normal(0, 10, 120)

demanda = tendencia + estacionalidad + ruido

df_demanda = pd.DataFrame({'Demanda_mm': demanda}, index=fechas)

# Partición Secuencial (Train: 8 años, Test: últimos 2 años)
train = df_demanda.iloc[:-24]
test = df_demanda.iloc[-24:]

print(f"Datos generados. Entrenamiento: {len(train)} meses, Validación: {len(test)} meses.")

### 1. Ajuste del Modelo Matemático (SARIMA)
Ajustaremos un modelo SARIMA. Estos modelos requieren especificar varios parámetros (p, d, q) para la parte normal y (P, D, Q, s) para la parte estacional.
Para este ejercicio en Python, definiremos parámetros clásicos para una serie mensual (ciclo $s=12$).

```

### Celda de Código 2

In [ ]:
# Inicializamos el modelo SARIMA
# (1, 1, 1) son los parámetros autorregresivos, de integración y de media móvil
# (1, 1, 1, 12) son los parámetros estacionales con un ciclo de 12 meses
modelo_sarima = SARIMAX(train['Demanda_mm'],
                        order=(1, 1, 1),
                        seasonal_order=(1, 1, 1, 12),
                        enforce_stationarity=False,
                        enforce_invertibility=False)

# Entrenamos el modelo con nuestros datos históricos
resultado_sarima = modelo_sarima.fit(disp=False)

# Mostramos el AIC del modelo (recuerda: mientras más bajo, mejor)
print(f"Modelo SARIMA entrenado. Criterio de Información de Akaike (AIC): {resultado_sarima.aic:.2f}")

### 2. Pronóstico y el Cono de Incertidumbre
Ahora le pediremos al modelo que pronostique los próximos 24 meses (el periodo de nuestro conjunto `test`).
También extraeremos el intervalo de confianza. Vas a notar cómo la banda se vuelve más ancha a medida que avanzamos en el tiempo, reflejando que es más difícil predecir diciembre de 2023 que enero de 2022.

```

### Celda de Código 3

In [ ]:
# Generamos el pronóstico para los siguientes 24 pasos (meses)
pronostico_obj = resultado_sarima.get_forecast(steps=24)

# Extraemos la predicción central (la línea media)
prediccion_central = pronostico_obj.predicted_mean

# Extraemos los intervalos de confianza al 95%
intervalos_confianza = pronostico_obj.conf_int(alpha=0.05)
limite_inferior = intervalos_confianza.iloc[:, 0]
limite_superior = intervalos_confianza.iloc[:, 1]

# Calculamos el error MAPE
def calcular_mape(y_real, y_pred):
    return np.mean(np.abs((y_real - y_pred) / y_real)) * 100

mape = calcular_mape(test['Demanda_mm'], prediccion_central)
print(f"Error de pronóstico (MAPE): {mape:.1f}%")

print("\nVista rápida del pronóstico vs realidad para los primeros 3 meses:")
comparacion = pd.DataFrame({
    'Realidad': test['Demanda_mm'][:3].values,
    'Pronóstico': prediccion_central[:3].values,
    'Límite_Inferior': limite_inferior[:3].values,
    'Límite_Superior': limite_superior[:3].values
}, index=test.index[:3]).round(1)
display(comparacion)

### 3. Visualización Profesional del Pronóstico
La mejor manera de comunicar resultados probabilísticos a tomadores de decisiones es mediante gráficos con bandas de sombra.

```

### Celda de Código 4

In [ ]:
plt.figure(figsize=(12, 5))

# 1. Graficamos el pasado (últimos 4 años del train para contexto)
plt.plot(train.index[-48:], train['Demanda_mm'].iloc[-48:], label='Histórico (Train)', color='black', lw=2)

# 2. Graficamos la realidad oculta (Test)
plt.plot(test.index, test['Demanda_mm'], label='Realidad Oculta (Test)', color='gray', linestyle='--')

# 3. Graficamos el pronóstico central
plt.plot(test.index, prediccion_central, label='Pronóstico SARIMA', color='blue', lw=2)

# 4. Sombreamos el intervalo de confianza (95%)
plt.fill_between(test.index, limite_inferior, limite_superior, color='blue', alpha=0.15, label='Intervalo de Confianza (95%)')

plt.title(f'Pronóstico de Demanda Hídrica - MAPE: {mape:.1f}%')
plt.xlabel('Fecha')
plt.ylabel('Demanda (mm/mes)')
plt.legend(loc='upper left')
plt.show()

### 🛑 Reflexión y Reserva Cognitiva (Síntesis manuscrita)
Toma tu cuaderno físico, observa la gráfica y responde:
1. Revisa la zona gris clara (la banda azul de incertidumbre). ¿Contiene en su interior a la línea gris punteada (la realidad) durante todo el periodo pronosticado? Si un punto real se saliera de la banda, ¿qué significaría estadísticamente?
2. Como ingeniero diseñando un reservorio de riego, si el pronóstico para agosto dice que la predicción central es 150 mm, pero el límite superior del intervalo es 190 mm. ¿Para cuál de los dos valores dimensionarías la capacidad del reservorio si quieres ser conservador y evitar un colapso?
3. ¿Por qué el intervalo de confianza se hace visualmente más ancho en el lado derecho de la gráfica (hacia el final de los 2 años)?

---

### Instrucciones para el reto en R (Trabajo Autónomo y Sesión 2)

**Misión:** Configurar los parámetros ARIMA (p,d,q) manualmente en Python es tedioso. La comunidad de R creó la función `auto.arima`, una de las herramientas más famosas en la ciencia de datos, capaz de buscar automáticamente el mejor modelo basándose en el AIC. Tu reto es aplicar esto en **Posit Cloud**.

**Pasos a seguir:**
1. Abre tu proyecto final en Posit Cloud y crea un documento RMarkdown (o Quarto).
2. Utiliza este *prompt* con tu asistente de IA (ChatGPT, Gemini, etc.):
   > *"Actúa como un profesor de hidrología estadística en R. En Python utilicé `statsmodels` para ajustar un modelo SARIMA a una serie de demanda hídrica, pero tuve que darle los parámetros (p,d,q) a mano. Necesito replicar el flujo en R utilizando el paquete `forecast`. Escribe código para simular la serie temporal, convertirla a objeto `ts`, y particionarla con `window()`. Luego, enséñame cómo usar la magia de `auto.arima()` sobre el conjunto de entrenamiento para que R encuentre el mejor modelo por sí solo, y cómo usar `forecast(modelo, h=24)` para predecir los próximos 24 meses. Finalmente, muestra cómo graficar esto maravillosamente con `autoplot()`, incluyendo las bandas de confianza. Explícalo paso a paso."*
3. Observa la salida de `auto.arima(train)` en la consola. Verás que te devuelve el modelo elegido (ej. `ARIMA(2,1,1)(1,1,0)[12]`). Compara el MAPE generado contra el de Python.
4. **Entrega y Preparación Final:** Renderiza tu documento. En tu "Bitácora de IA", redacta tus aprendizajes finales sobre modelación predictiva.
¡Felicitaciones! Has completado las bases técnicas. La Semana 16 estará dedicada 100% a la sustentación de tu Proyecto Integrador.